In [ ]:
import os
import pathlib
import datetime
import pandas as pd

# --- Configure folders to scan ---
FOLDERS = [
    #"/Users/kirillmaier/DocLocal/zzz_archive/Астана",
    "/Users/kirillmaier/DocLocal",
    "/Users/kirillmaier/Documents",
    "/Users/kirillmaier/Library/CloudStorage/MountainDuck-Krll2/bak_1tb",
    "/Users/kirillmaier/Library/CloudStorage/MountainDuck-Krll",
    # "/Users/kirillmaier/DocLocal/another_folder",  # add more folders here
]

RECURSIVE = True  # set False to scan top-level only


In [ ]:
def format_int_spaces(value):
    return f"{int(value):,}".replace(",", " ")


def format_float_spaces(value, digits=2):
    formatted = f"{float(value):,.{digits}f}"
    return formatted.replace(",", " ")


def format_dt(value):
    return value.strftime("%Y-%m-%d %H:%M")


def collect_files(folders, recursive=True):
    records = []
    scanned_count = 0

    for folder in folders:
        base = pathlib.Path(folder)
        if not base.exists():
            print(f"WARNING: Folder not found: {folder}")
            continue

        pattern = "**/*" if recursive else "*"
        for path in sorted(base.glob(pattern)):
            if not path.is_file():
                continue

            scanned_count += 1

            stat = path.stat()

            try:
                created_dt = datetime.datetime.fromtimestamp(stat.st_birthtime)
            except AttributeError:
                created_dt = datetime.datetime.fromtimestamp(stat.st_mtime)

            modified_dt = datetime.datetime.fromtimestamp(stat.st_mtime)
            records.append({
                "folder": str(base),
                "relative_path": str(path.relative_to(base)),
                "name": path.name,
                "extension": path.suffix.lower() if path.suffix else "(none)",
                "size_bytes": stat.st_size,
                "size_kb": round(stat.st_size / 1024, 2),
                "created_dt": created_dt,
                "modified_dt": modified_dt,
                "created": format_dt(created_dt),
                "modified": format_dt(modified_dt),
            })

    return pd.DataFrame(records)


df = collect_files(FOLDERS, recursive=RECURSIVE)
print(f"Total files found: {format_int_spaces(len(df))}")

In [ ]:
# --- Statistics by creation month/year ---
df_stats = df.copy()
df_stats["created_month"] = df_stats["created_dt"].dt.to_period("M").astype(str)

monthly_stats = (
    df_stats.groupby("created_month")
    .agg(
        file_count=("name", "count"),
        total_size_bytes=("size_bytes", "sum"),
        avg_size_bytes=("size_bytes", "mean"),
        earliest_created=("created_dt", "min"),
        latest_created=("created_dt", "max"),
    )
    .reset_index()
    .sort_values("created_month", ascending=False)
)

current_month = datetime.datetime.now().strftime("%Y-%m")
current_month_stats = monthly_stats[monthly_stats["created_month"] == current_month].copy()

monthly_stats_display = monthly_stats.copy()
monthly_stats_display["file_count"] = monthly_stats_display["file_count"].map(format_int_spaces)
monthly_stats_display["total_size_bytes"] = monthly_stats_display["total_size_bytes"].map(format_int_spaces)
monthly_stats_display["avg_size_bytes"] = monthly_stats_display["avg_size_bytes"].map(lambda value: format_float_spaces(value, 0))
monthly_stats_display["earliest_created"] = monthly_stats_display["earliest_created"].map(format_dt)
monthly_stats_display["latest_created"] = monthly_stats_display["latest_created"].map(format_dt)

total_bytes = df["size_bytes"].sum()
print(
    f"Total size: {format_int_spaces(total_bytes)} bytes  |  "
    f"Files: {format_int_spaces(len(df))}  |  "
    f"Months: {format_int_spaces(len(monthly_stats))}"
)
print(f"Current month: {current_month}")

if current_month_stats.empty:
    print("Current month stats: 0 files")
else:
    month_row = current_month_stats.iloc[0]
    print(
        "Current month stats: "
        f"files={format_int_spaces(month_row['file_count'])}, "
        f"total_size_bytes={format_int_spaces(month_row['total_size_bytes'])}, "
        f"avg_size_bytes={format_float_spaces(month_row['avg_size_bytes'], 0)}"
    )

display(monthly_stats_display)


In [ ]:
# --- Full index sorted by creation date (newest first) ---
df_sorted = df.sort_values("created_dt", ascending=False).reset_index(drop=True)

df_display = df_sorted.copy()
df_display["size_bytes"] = df_display["size_bytes"].map(format_int_spaces)
df_display["size_kb"] = df_display["size_kb"].map(lambda value: format_float_spaces(value, 2))
df_display = df_display[[
    "folder",
    "relative_path",
    "name",
    "extension",
    "size_bytes",
    "size_kb",
    "created",
    "modified",
]]

# Uncomment to export raw data:
# df_sorted.to_csv("/Users/kirillmaier/DocLocal/files_index.csv", index=False)

#display(df_display)


In [ ]:
# --- Timeline of file creation ---
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

creation_timeline = df.copy().sort_values("created_dt")
creation_timeline["created_day"] = creation_timeline["created_dt"].dt.floor("D")

daily_created = (
    creation_timeline.groupby("created_day")
    .size()
    .reset_index(name="file_count")
)

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(
    daily_created["created_day"],
    daily_created["file_count"],
    marker="o",
    linewidth=2,
    color="#1f77b4",
    label="Files created per day",
)
ax.bar(
    daily_created["created_day"],
    daily_created["file_count"],
    width=0.8,
    alpha=0.2,
    color="#1f77b4",
)

ax.set_title("Files creation timeline")
ax.set_xlabel("Creation date")
ax.set_ylabel("Files created")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
fig.autofmt_xdate(rotation=30)
ax.legend(loc="upper left")

plt.tight_layout()
plt.show()

display(daily_created)


In [ ]:
# --- Fast duplicate files finder (same content) ---
import hashlib


def file_sha256(path, chunk_size=1024 * 1024):
    hasher = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            hasher.update(chunk)
    return hasher.hexdigest()


def head_sha256(path, head_bytes=64 * 1024):
    hasher = hashlib.sha256()
    with open(path, "rb") as f:
        hasher.update(f.read(head_bytes))
    return hasher.hexdigest()


dup_df = df.copy()
dup_df["full_path"] = dup_df.apply(
    lambda row: str(pathlib.Path(row["folder"]) / row["relative_path"]),
    axis=1,
)

# Stage 1: only same-size files can be true duplicates
size_candidates = dup_df[dup_df.duplicated("size_bytes", keep=False)].copy()
print(f"Same-size candidate files: {format_int_spaces(len(size_candidates))}")

if size_candidates.empty:
    duplicate_files = pd.DataFrame()
    print("No duplicate files found.")
else:
    # Stage 2: hash only first 64KB to reduce expensive full-file hashing
    size_candidates["head_hash"] = size_candidates["full_path"].map(head_sha256)
    quick_candidates = size_candidates[
        size_candidates.duplicated(["size_bytes", "head_hash"], keep=False)
    ].copy()
    print(f"Quick-hash candidate files: {format_int_spaces(len(quick_candidates))}")

    if quick_candidates.empty:
        duplicate_files = pd.DataFrame()
        print("No duplicate files found.")
    else:
        # Stage 3: full hash only for remaining candidates
        quick_candidates["full_hash"] = quick_candidates["full_path"].map(file_sha256)
        duplicate_files = quick_candidates[
            quick_candidates.duplicated(["size_bytes", "full_hash"], keep=False)
        ].copy()

        if duplicate_files.empty:
            print("No duplicate files found.")
        else:
            duplicate_files = duplicate_files.sort_values(
                ["size_bytes", "full_hash", "full_path"],
                ascending=[False, True, True],
            ).reset_index(drop=True)
            duplicate_files["duplicate_group"] = (
                duplicate_files.groupby(["size_bytes", "full_hash"]).ngroup() + 1
            )
            duplicate_files["copies_in_group"] = duplicate_files.groupby(
                ["size_bytes", "full_hash"]
            )["full_path"].transform("count")

            duplicate_files_display = duplicate_files[[
                "duplicate_group",
                "copies_in_group",
                "size_bytes",
                "created",
                "full_path",
            ]].copy()
            duplicate_files_display["size_bytes"] = duplicate_files_display["size_bytes"].map(format_int_spaces)

            wasted_bytes = (
                duplicate_files.groupby(["size_bytes", "full_hash"])["size_bytes"]
                .first()
                .mul(
                    duplicate_files.groupby(["size_bytes", "full_hash"])["full_path"].count() - 1
                )
                .sum()
            )

            print(
                f"Duplicate groups: {format_int_spaces(duplicate_files['duplicate_group'].nunique())}  |  "
                f"Duplicate files: {format_int_spaces(len(duplicate_files))}  |  "
                f"Wasted bytes: {format_int_spaces(wasted_bytes)}"
            )
            display(duplicate_files_display)


In [ ]:
# --- Top 20 heaviest files ---
top20_heavy = df.copy().sort_values("size_bytes", ascending=False).head(20).reset_index(drop=True)
top20_heavy["full_path"] = top20_heavy.apply(
    lambda row: str(pathlib.Path(row["folder"]) / row["relative_path"]),
    axis=1,
)

top20_heavy_display = top20_heavy[[
    "size_bytes",
    "size_kb",
    "created",
    "modified",
    "extension",
    "full_path",
]].copy()

top20_heavy_display["size_bytes"] = top20_heavy_display["size_bytes"].map(format_int_spaces)
top20_heavy_display["size_kb"] = top20_heavy_display["size_kb"].map(lambda value: format_float_spaces(value, 2))

print(f"Top heavy files shown: {format_int_spaces(len(top20_heavy_display))}")
display(top20_heavy_display)


In [ ]:
# --- Biggest equal directory sets across all levels (names ignored) ---
import hashlib
from collections import defaultdict


def file_sha256_cached(path, cache, chunk_size=1024 * 1024):
    key = str(path)
    if key in cache:
        return cache[key]

    hasher = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            hasher.update(chunk)

    digest = hasher.hexdigest()
    cache[key] = digest
    return digest


def digest_from_parts(parts):
    hasher = hashlib.sha256()
    for part in parts:
        hasher.update(part.encode("utf-8"))
        hasher.update(b"|")
    return hasher.hexdigest()


all_dirs = []
for root in FOLDERS:
    root_path = pathlib.Path(root)
    if not root_path.exists() or not root_path.is_dir():
        print(f"WARNING: Root folder not found: {root}")
        continue

    all_dirs.append(root_path)
    all_dirs.extend([p for p in root_path.rglob("*") if p.is_dir()])

# De-duplicate paths if configured roots overlap.
all_dirs = sorted({p.resolve() for p in all_dirs})
print(f"Directories scanned: {format_int_spaces(len(all_dirs))}")

if not all_dirs:
    equal_dir_groups = pd.DataFrame()
    equal_dir_members = pd.DataFrame()
    print("No directories found under selected roots.")
else:
    file_hash_cache = {}
    dir_signature = {}
    dir_file_count = {}
    dir_total_size = {}

    # Bottom-up: if children are equal, parents can also become equal.
    dirs_bottom_up = sorted(all_dirs, key=lambda p: len(p.parts), reverse=True)

    for d in dirs_bottom_up:
        direct_files = [p for p in d.iterdir() if p.is_file()]
        direct_subdirs = [p for p in d.iterdir() if p.is_dir()]

        file_parts = []
        files_count = 0
        total_size = 0
        for f in sorted(direct_files):
            size = f.stat().st_size
            sha = file_sha256_cached(f, file_hash_cache)
            file_parts.append(f"F:{size}:{sha}")
            files_count += 1
            total_size += size

        child_parts = []
        for child in sorted(direct_subdirs):
            child = child.resolve()
            child_sig = dir_signature.get(child, "")
            child_parts.append(f"D:{child_sig}")
            files_count += dir_file_count.get(child, 0)
            total_size += dir_total_size.get(child, 0)

        # Names are ignored: compare as a set of files/subfolders by content.
        signature = digest_from_parts(sorted(file_parts) + sorted(child_parts))
        dir_signature[d] = signature
        dir_file_count[d] = files_count
        dir_total_size[d] = total_size

    signature_groups = defaultdict(list)
    for d in all_dirs:
        signature_groups[dir_signature[d]].append(d)

    group_rows = []
    for sig, dirs in signature_groups.items():
        if len(dirs) < 2:
            continue

        sample = dirs[0]
        files_in_tree = dir_file_count[sample]
        total_size_bytes = dir_total_size[sample]

        # Skip empty-only groups to focus on meaningful equal sets.
        if files_in_tree == 0:
            continue

        group_rows.append({
            "content_signature": sig,
            "folders_in_group": len(dirs),
            "files_in_tree": files_in_tree,
            "total_size_bytes": total_size_bytes,
            "members": [str(x) for x in sorted(dirs)],
        })

    if not group_rows:
        equal_dir_groups = pd.DataFrame()
        equal_dir_members = pd.DataFrame()
        print("No fully equal directory sets found.")
    else:
        equal_dir_groups = pd.DataFrame(group_rows).sort_values(
            ["total_size_bytes", "folders_in_group", "files_in_tree"],
            ascending=[False, False, False],
        ).reset_index(drop=True)
        equal_dir_groups["equal_set_id"] = equal_dir_groups.index + 1

        member_rows = []
        for _, row in equal_dir_groups.iterrows():
            for folder in row["members"]:
                member_rows.append({
                    "equal_set_id": row["equal_set_id"],
                    "folders_in_group": row["folders_in_group"],
                    "files_in_tree": row["files_in_tree"],
                    "total_size_bytes": row["total_size_bytes"],
                    "parent_folder": str(pathlib.Path(folder).parent),
                    "folder": folder,
                })

        equal_dir_members = pd.DataFrame(member_rows).sort_values(
            ["total_size_bytes", "equal_set_id", "folder"],
            ascending=[False, True, True],
        ).reset_index(drop=True)

        groups_display = equal_dir_groups[[
            "equal_set_id",
            "folders_in_group",
            "files_in_tree",
            "total_size_bytes",
        ]].copy()
        groups_display["folders_in_group"] = groups_display["folders_in_group"].map(format_int_spaces)
        groups_display["files_in_tree"] = groups_display["files_in_tree"].map(format_int_spaces)
        groups_display["total_size_bytes"] = groups_display["total_size_bytes"].map(format_int_spaces)

        members_display = equal_dir_members[[
            "equal_set_id",
            "folders_in_group",
            "files_in_tree",
            "total_size_bytes",
            "parent_folder",
            "folder",
        ]].copy()
        members_display["folders_in_group"] = members_display["folders_in_group"].map(format_int_spaces)
        members_display["files_in_tree"] = members_display["files_in_tree"].map(format_int_spaces)
        members_display["total_size_bytes"] = members_display["total_size_bytes"].map(format_int_spaces)

        print(
            f"Equal sets found: {format_int_spaces(len(equal_dir_groups))}  |  "
            f"Folders matched: {format_int_spaces(len(equal_dir_members))}"
        )
        display(groups_display)
        display(members_display)


In [ ]:
members_display.to_csv("Biggest equal directory sets across all levels (names ignored).csv", index=False)